# DreamerV3 World Model MuJoCo Humanoid (CuPy GPU)
From-scratch DreamerV3 with CuPy (GPU arrays) for full GPU-accelerated forward+backward.
All autograd, neural net layers, RSSM, training logic is our code — CuPy just replaces NumPy as the array backend.

In [ ]:
!nvidia-smi
!pip install cupy-cuda12x gymnasium[mujoco] -q
import os
os.environ['MUJOCO_GL'] = 'egl'
import cupy as cp
print(f'CuPy version: {cp.__version__}')
print(f'GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"]}')
a = cp.random.randn(512, 512, dtype=cp.float32)
b = cp.random.randn(512, 512, dtype=cp.float32)
_ = cp.matmul(a, b)
print('CuPy GPU matmul works!')

In [ ]:
import os
for d in ['nn', 'model', 'agent', 'training']:
    os.makedirs(d, exist_ok=True)
    open(f'{d}/__init__.py', 'w').close()
print('Directories created')

In [ ]:
%%writefile nn/tensor.py
import numpy as np
try:
    import cupy as cp
    _xp = cp
    GPU = True
except ImportError:
    _xp = np
    GPU = False

def to_gpu(arr):
    if GPU and isinstance(arr, np.ndarray): return cp.asarray(arr, dtype=cp.float32)
    return arr

def to_cpu(arr):
    if GPU and isinstance(arr, cp.ndarray): return cp.asnumpy(arr)
    return np.asarray(arr)

def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, (g, s) in enumerate(zip(grad.shape, shape)):
        if s == 1 and g != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        if isinstance(data, Tensor):
            data = data.data
        if isinstance(data, (float, int)):
            self.data = _xp.array(data, dtype=_xp.float32)
        elif isinstance(data, np.ndarray):
            self.data = to_gpu(data.astype(np.float32))
        elif GPU and isinstance(data, cp.ndarray):
            self.data = data if data.dtype == cp.float32 else data.astype(cp.float32)
        else:
            self.data = _xp.array(data, dtype=_xp.float32)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._children = set(_children)
        self._op = _op

    @property
    def shape(self):
        return self.data.shape

    def __float__(self):
        return float(to_cpu(self.data))

    def __int__(self):
        return int(to_cpu(self.data))

    def zero_grad(self):
        self.grad = None

    def detach(self):
        return Tensor(self.data.copy(), requires_grad=False)

    def numpy(self):
        return to_cpu(self.data)

    def backward(self):
        topo = []
        visited = set()
        def build(node):
            if node not in visited:
                visited.add(node)
                for child in node._children:
                    build(child)
                topo.append(node)
        build(self)
        self.grad = _xp.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='+')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __radd__(self, other): return self.__add__(other)

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data - other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='-')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(-out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rsub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other.__sub__(self)

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='*')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad * other.data, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad * self.data, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rmul__(self, other): return self.__mul__(other)
    def __neg__(self): return self * (-1)

    def __pow__(self, power):
        out = Tensor(self.data ** power, requires_grad=self.requires_grad, _children=(self,), _op=f'**{power}')
        def _backward():
            if self.requires_grad:
                g = out.grad * power * self.data ** (power - 1)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def matmul(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(_xp.matmul(self.data, other.data), requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='matmul')
        def _backward():
            if self.requires_grad:
                g = _xp.matmul(out.grad, _xp.swapaxes(other.data, -1, -2))
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _xp.matmul(_xp.swapaxes(self.data, -1, -2), out.grad)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), requires_grad=self.requires_grad, _children=(self,), _op='sum')
        def _backward():
            if self.requires_grad:
                if axis is None:
                    g = _xp.ones_like(self.data) * out.grad
                elif not keepdims:
                    g = _xp.expand_dims(out.grad, axis=axis) * _xp.ones_like(self.data)
                else:
                    g = out.grad * _xp.ones_like(self.data)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def mean(self, axis=None, keepdims=False):
        n = self.data.size if axis is None else self.data.shape[axis]
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)

    def reshape(self, *shape):
        if len(shape) == 1 and isinstance(shape[0], tuple): shape = shape[0]
        out = Tensor(self.data.reshape(shape), requires_grad=self.requires_grad, _children=(self,), _op='reshape')
        def _backward():
            if self.requires_grad:
                self.grad = out.grad.reshape(self.shape) if self.grad is None else self.grad + out.grad.reshape(self.shape)
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + _xp.exp(-_xp.clip(self.data, -88, 88)))
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='sigmoid')
        def _backward():
            if self.requires_grad:
                g = out.grad * s * (1 - s)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def tanh(self):
        t = _xp.tanh(self.data)
        out = Tensor(t, requires_grad=self.requires_grad, _children=(self,), _op='tanh')
        def _backward():
            if self.requires_grad:
                g = out.grad * (1 - t ** 2)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def exp(self):
        e = _xp.exp(_xp.clip(self.data, -88, 88))
        out = Tensor(e, requires_grad=self.requires_grad, _children=(self,), _op='exp')
        def _backward():
            if self.requires_grad:
                g = out.grad * e
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(_xp.maximum(0, self.data), requires_grad=self.requires_grad, _children=(self,), _op='relu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (self.data > 0).astype(_xp.float32)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def silu(self):
        s = 1.0 / (1.0 + _xp.exp(-_xp.clip(self.data, -88, 88)))
        out = Tensor(self.data * s, requires_grad=self.requires_grad, _children=(self,), _op='silu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (s + self.data * s * (1 - s))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def softmax(self, axis=-1):
        e = _xp.exp(self.data - self.data.max(axis=axis, keepdims=True))
        s = e / e.sum(axis=axis, keepdims=True)
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='softmax')
        def _backward():
            if self.requires_grad:
                g = out.grad * s - s * (out.grad * s).sum(axis=axis, keepdims=True)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def log(self):
        out = Tensor(_xp.log(self.data + 1e-8), requires_grad=self.requires_grad, _children=(self,), _op='log')
        def _backward():
            if self.requires_grad:
                g = out.grad / (self.data + 1e-8)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def clamp(self, min_val=None, max_val=None):
        data = self.data.copy()
        if min_val is not None: data = _xp.maximum(data, min_val)
        if max_val is not None: data = _xp.minimum(data, max_val)
        out = Tensor(data, requires_grad=self.requires_grad, _children=(self,), _op='clamp')
        def _backward():
            if self.requires_grad:
                mask = _xp.ones_like(self.data)
                if min_val is not None: mask *= (self.data >= min_val)
                if max_val is not None: mask *= (self.data <= max_val)
                g = out.grad * mask
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def symlog(self):
        out = Tensor(_xp.sign(self.data) * _xp.log(1 + _xp.abs(self.data)), requires_grad=self.requires_grad, _children=(self,), _op='symlog')
        def _backward():
            if self.requires_grad:
                g = out.grad / (1 + _xp.abs(self.data))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def straight_through(self, hard):
        if isinstance(hard, np.ndarray): hard = to_gpu(hard)
        out = Tensor(hard, requires_grad=self.requires_grad, _children=(self,), _op='st')
        def _backward():
            if self.requires_grad:
                self.grad = out.grad.copy() if self.grad is None else self.grad + out.grad
        out._backward = _backward
        return out

    def layer_norm(self, scale, shift, eps=1e-5):
        mu = self.data.mean(axis=-1, keepdims=True)
        var = self.data.var(axis=-1, keepdims=True)
        x_hat = (self.data - mu) / _xp.sqrt(var + eps)
        y = scale.data * x_hat + shift.data
        out = Tensor(y, requires_grad=self.requires_grad or scale.requires_grad or shift.requires_grad, _children=(self, scale, shift), _op='ln')
        def _backward():
            if out.grad is None: return
            D = self.data.shape[-1]
            inv_std = 1.0 / _xp.sqrt(var + eps)
            dy = out.grad
            if scale.requires_grad:
                sg = (dy * x_hat).sum(axis=tuple(range(dy.ndim - 1)))
                scale.grad = sg if scale.grad is None else scale.grad + sg
            if shift.requires_grad:
                bg = dy.sum(axis=tuple(range(dy.ndim - 1)))
                shift.grad = bg if shift.grad is None else shift.grad + bg
            if self.requires_grad:
                dx_hat = dy * scale.data
                g = inv_std * (dx_hat - dx_hat.mean(axis=-1, keepdims=True) - x_hat * (dx_hat * x_hat).mean(axis=-1, keepdims=True))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out


def cat(tensors, axis=0):
    data = _xp.concatenate([t.data for t in tensors], axis=axis)
    out = Tensor(data, requires_grad=any(t.requires_grad for t in tensors), _children=tuple(tensors), _op='cat')
    def _backward():
        if out.grad is None: return
        sections = _xp.cumsum(_xp.array([t.shape[axis] for t in tensors[:-1]]))
        grads = _xp.split(out.grad, to_cpu(sections).astype(int).tolist(), axis=axis)
        for t, g in zip(tensors, grads):
            if t.requires_grad:
                t.grad = g.copy() if t.grad is None else t.grad + g
    out._backward = _backward
    return out


def one_hot(indices, num_classes):
    flat = indices.reshape(-1)
    oh = np.zeros((flat.size, num_classes), dtype=np.float32)
    oh[np.arange(flat.size), flat] = 1.0
    return oh.reshape(indices.shape + (num_classes,))

# Twohot encoding for DreamerV3 reward/value prediction
NUM_BINS = 255
SYMLOG_MIN, SYMLOG_MAX = -20.0, 20.0
BINS = _xp.linspace(SYMLOG_MIN, SYMLOG_MAX, NUM_BINS).astype(_xp.float32)

def symexp(x):
    return _xp.sign(x) * (_xp.exp(_xp.abs(x)) - 1)

def twohot_encode(x):
    x = _xp.clip(x, SYMLOG_MIN, SYMLOG_MAX)
    k = np.searchsorted(to_cpu(BINS) if GPU else BINS, to_cpu(x) if GPU else x).astype(np.int64)
    if GPU: k = _xp.asarray(k)
    k = _xp.clip(k, 1, NUM_BINS - 1)
    below = k - 1
    weight_above = (x - BINS[below]) / (BINS[k] - BINS[below] + 1e-8)
    weight_below = 1.0 - weight_above
    shape = x.shape + (NUM_BINS,)
    oh = _xp.zeros(shape, dtype=_xp.float32)
    flat_oh = oh.reshape(-1, NUM_BINS)
    flat_below = below.reshape(-1)
    flat_above = k.reshape(-1)
    idx = _xp.arange(flat_oh.shape[0])
    flat_oh[idx, flat_below] = weight_below.reshape(-1)
    flat_oh[idx, flat_above] = weight_above.reshape(-1)
    return oh

def twohot_loss(logits, target_scalar):
    target_symlog = _xp.sign(target_scalar) * _xp.log(1 + _xp.abs(target_scalar))
    target_oh = twohot_encode(target_symlog)
    log_probs = logits.softmax(axis=-1).log()
    loss = (Tensor(target_oh) * log_probs * (-1.0)).sum(axis=-1)
    return loss

def twohot_decode(logits):
    probs = logits.softmax(axis=-1)
    decoded_symlog = (probs.data * BINS).sum(axis=-1, keepdims=True)
    return Tensor(symexp(decoded_symlog))

SYMEXP_BINS = symexp(BINS)

def twohot_decode_diff(logits):
    """Differentiable decode: gradient flows logits -> softmax -> weighted sum -> actor."""
    probs = logits.softmax(axis=-1)
    return (probs * Tensor(SYMEXP_BINS)).sum(axis=-1, keepdims=True)

In [ ]:
%%writefile nn/linear.py
import numpy as np
from nn.tensor import Tensor, _xp

class Linear:
    def __init__(self, in_features, out_features, bias=True):
        scale = (2.0 / in_features) ** 0.5
        self.weight = Tensor(_xp.random.randn(in_features, out_features).astype(_xp.float32) * scale, requires_grad=True)
        self.bias = Tensor(_xp.zeros(out_features, dtype=_xp.float32), requires_grad=True) if bias else None

    def __call__(self, x):
        out = x.matmul(self.weight)
        if self.bias is not None: out = out + self.bias
        return out

    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias else [])

class LayerNorm:
    def __init__(self, dim):
        self.scale = Tensor(_xp.ones(dim, dtype=_xp.float32), requires_grad=True)
        self.shift = Tensor(_xp.zeros(dim, dtype=_xp.float32), requires_grad=True)
    def __call__(self, x):
        return x.layer_norm(self.scale, self.shift)
    def parameters(self):
        return [self.scale, self.shift]

In [ ]:
%%writefile nn/mlp.py
from nn.linear import Linear, LayerNorm

class MLP:
    def __init__(self, sizes, activation='silu'):
        self.activation = activation
        self.layers = [Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]
        self.norms = [LayerNorm(sizes[i+1]) for i in range(len(sizes)-2)]
    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = x.silu() if self.activation == 'silu' else x.relu()
        return x
    def parameters(self):
        p = []
        for l in self.layers: p.extend(l.parameters())
        for n in self.norms: p.extend(n.parameters())
        return p

In [ ]:
%%writefile nn/optimizer.py
from nn.tensor import _xp

class AdamW:
    def __init__(self, parameters, lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01, grad_clip=100.0):
        self.parameters = parameters
        self.lr, self.eps, self.weight_decay, self.grad_clip = lr, eps, weight_decay, grad_clip
        self.beta1, self.beta2 = betas
        self.t = 0
        self.m = [_xp.zeros_like(p.data) for p in parameters]
        self.v = [_xp.zeros_like(p.data) for p in parameters]

    def step(self):
        self.t += 1
        if self.grad_clip > 0:
            norm = float(_xp.sqrt(sum(_xp.sum(p.grad**2) for p in self.parameters if p.grad is not None)))
            if norm > self.grad_clip:
                for p in self.parameters:
                    if p.grad is not None: p.grad = p.grad * (self.grad_clip / (norm + 1e-6))
        for i, p in enumerate(self.parameters):
            if p.grad is None: continue
            self.m[i] = self.beta1 * self.m[i] + (1-self.beta1) * p.grad
            self.v[i] = self.beta2 * self.v[i] + (1-self.beta2) * (p.grad**2)
            mh = self.m[i] / (1-self.beta1**self.t)
            vh = self.v[i] / (1-self.beta2**self.t)
            p.data -= self.lr * self.weight_decay * p.data
            p.data -= self.lr * mh / (_xp.sqrt(vh) + self.eps)

    def zero_grad(self):
        for p in self.parameters: p.zero_grad()

In [ ]:
%%writefile nn/gru_cell.py
import numpy as np
from nn.tensor import Tensor, cat, _xp, GPU, to_cpu
from nn.linear import Linear

# Module-level flag: set True to activate fused kernel path during inference
INFERENCE_MODE = False

# Fused GRU element-wise kernel: computes gates + candidate + output in one launch
_fused_gru_kernel = None
if GPU:
    import cupy as cp
    _fused_gru_kernel = cp.RawKernel(r'''
    extern "C" __global__
    void fused_gru_elementwise(
        const float* rz_out,    // (batch, 2*H) = concat of reset and update pre-activations
        const float* h_prev,    // (batch, H)
        const float* x_data,    // (batch, input_size)
        const float* Wc,        // (input_size+H, H)
        const float* bc,        // (H,)
        float* h_new,           // (batch, H) output
        float* r_out,           // (batch, H) reset gate (for backward)
        float* z_out,           // (batch, H) update gate (for backward)
        float* cand_out,        // (batch, H) candidate (for backward)
        int batch, int input_size, int H
    ) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        int total = batch * H;
        if (idx >= total) return;

        int b = idx / H;
        int j = idx % H;

        // Sigmoid for reset and update gates
        float r_pre = rz_out[b * (2*H) + j];
        float z_pre = rz_out[b * (2*H) + H + j];
        float r = 1.0f / (1.0f + expf(-r_pre));
        float z = 1.0f / (1.0f + expf(-z_pre));
        float h_p = h_prev[b * H + j];

        // Candidate: dot product of [x, r*h] with Wc column j, then tanh
        float cand_pre = bc[j];
        int combined_size = input_size + H;
        for (int k = 0; k < input_size; k++) {
            cand_pre += x_data[b * input_size + k] * Wc[k * H + j];
        }
        for (int k = 0; k < H; k++) {
            float rh = (k == j) ? r * h_p : (1.0f / (1.0f + expf(-rz_out[b*(2*H) + k]))) * h_prev[b*H + k];
            cand_pre += rh * Wc[(input_size + k) * H + j];
        }
        float cand = tanhf(cand_pre);

        // Output: z * h_prev + (1 - z) * candidate
        float h_out = z * h_p + (1.0f - z) * cand;

        h_new[idx] = h_out;
        r_out[idx] = r;
        z_out[idx] = z;
        cand_out[idx] = cand;
    }
    ''', 'fused_gru_elementwise')


class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.reset_gate = Linear(input_size + hidden_size, hidden_size)
        self.update_gate = Linear(input_size + hidden_size, hidden_size)
        self.candidate = Linear(input_size + hidden_size, hidden_size)

    def __call__(self, x, h):
        if h is None: h = Tensor(_xp.zeros((x.shape[0], self.hidden_size), dtype=_xp.float32))
        H = self.hidden_size
        batch = x.shape[0]

        if _fused_gru_kernel is not None and INFERENCE_MODE:
            # FAST PATH: fused kernel for inference (no gradient needed)
            combined = _xp.concatenate([x.data, h.data], axis=1)
            # Compute reset+update pre-activations in one matmul
            W_rz = _xp.concatenate([self.reset_gate.weight.data, self.update_gate.weight.data], axis=1)
            b_rz = _xp.concatenate([self.reset_gate.bias.data, self.update_gate.bias.data])
            rz_pre = _xp.matmul(combined, W_rz) + b_rz  # (batch, 2*H)

            h_new = _xp.empty((batch, H), dtype=_xp.float32)
            r_buf = _xp.empty((batch, H), dtype=_xp.float32)
            z_buf = _xp.empty((batch, H), dtype=_xp.float32)
            c_buf = _xp.empty((batch, H), dtype=_xp.float32)

            threads = 256
            blocks = (batch * H + threads - 1) // threads
            _fused_gru_kernel((blocks,), (threads,), (
                rz_pre, h.data, x.data,
                self.candidate.weight.data, self.candidate.bias.data,
                h_new, r_buf, z_buf, c_buf,
                batch, self.input_size, H
            ))
            return Tensor(h_new)

        # TRAINING PATH: standard ops with autograd tracking
        combined = cat([x, h], axis=1)
        r = self.reset_gate(combined).sigmoid()
        z = self.update_gate(combined).sigmoid()
        candidate = self.candidate(cat([x, r * h], axis=1)).tanh()
        return z * h + (1 - z) * candidate

    def parameters(self):
        p = []
        for l in [self.reset_gate, self.update_gate, self.candidate]: p.extend(l.parameters())
        return p

In [ ]:
%%writefile model/encoder.py
from nn.mlp import MLP

class Encoder:
    def __init__(self, obs_size, hidden_size, latent_size):
        self.net = MLP([obs_size, hidden_size, hidden_size, latent_size])
    def __call__(self, obs): return self.net(obs)
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/decoder.py
from nn.mlp import MLP
from nn.tensor import cat
class Decoder:
    def __init__(self, state_size, obs_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, obs_size])
    def __call__(self, h, z): return self.net(cat([h, z], axis=1))
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/reward_model.py
from nn.mlp import MLP
from nn.tensor import cat, NUM_BINS, twohot_decode, twohot_decode_diff
class RewardModel:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, NUM_BINS])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1))
    def predict(self, h, z):
        logits = self(h, z)
        return twohot_decode(logits)
    def predict_diff(self, h, z):
        logits = self(h, z)
        return twohot_decode_diff(logits)
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/continue_model.py
from nn.mlp import MLP
from nn.tensor import cat
class ContinueModel:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z): return self.net(cat([h, z], axis=1)).sigmoid()
    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile model/rssm.py
import numpy as np
from nn.tensor import Tensor, cat, one_hot, _xp, to_cpu
from nn.linear import Linear
from nn.mlp import MLP
from nn.gru_cell import GRUCell

class RSSM:
    def __init__(self, obs_size, action_size, hidden_size, stoch_size, stoch_classes):
        self.hidden_size = hidden_size
        self.stoch_size = stoch_size
        self.stoch_classes = stoch_classes
        stoch_flat = stoch_size * stoch_classes
        self.pre_gru = Linear(stoch_flat + action_size, hidden_size)
        self.gru = GRUCell(hidden_size, hidden_size)
        self.prior = MLP([hidden_size, hidden_size, stoch_flat])
        self.posterior = MLP([hidden_size + obs_size, hidden_size, stoch_flat])

    def forward(self, prev_h, prev_z, action, encoded_obs=None):
        if prev_z is None: prev_z = Tensor(_xp.zeros((action.shape[0], self.stoch_size * self.stoch_classes), dtype=_xp.float32))
        if prev_h is None: prev_h = Tensor(_xp.zeros((action.shape[0], self.hidden_size), dtype=_xp.float32))
        gru_input = self.pre_gru(cat([prev_z, action], axis=1)).silu()
        h = self.gru(gru_input, prev_h)
        prior_logits = self.prior(h).reshape(-1, self.stoch_size, self.stoch_classes)
        if encoded_obs is not None:
            post_logits = self.posterior(cat([h, encoded_obs], axis=1)).reshape(-1, self.stoch_size, self.stoch_classes)
            logits = post_logits
        else:
            post_logits = None
            logits = prior_logits
        probs = logits.softmax(axis=-1)
        # Uniform mix: 99% network + 1% uniform (prevents near-deterministic categoricals)
        uniform = _xp.ones_like(probs.data) / self.stoch_classes
        mixed = 0.99 * probs.data + 0.01 * uniform
        probs = probs.straight_through(mixed)
        probs_np = to_cpu(probs.data)
        indices = np.array([[np.random.choice(self.stoch_classes, p=probs_np[b,d]) for d in range(self.stoch_size)] for b in range(probs_np.shape[0])])
        z = probs.straight_through(one_hot(indices, self.stoch_classes))
        return h, z.reshape(-1, self.stoch_size * self.stoch_classes), prior_logits, post_logits

    def parameters(self):
        p = self.pre_gru.parameters()
        p.extend(self.gru.parameters())
        p.extend(self.prior.parameters())
        p.extend(self.posterior.parameters())
        return p

In [ ]:
%%writefile model/world_model.py
from nn.tensor import Tensor, cat, twohot_loss, _xp
from model.encoder import Encoder
from model.rssm import RSSM
from model.decoder import Decoder
from model.reward_model import RewardModel
from model.continue_model import ContinueModel

class WorldModel:
    def __init__(self, obs_size, action_size, hidden_size, stoch_size, stoch_classes):
        state_size = hidden_size + stoch_size * stoch_classes
        self.encoder = Encoder(obs_size, hidden_size, hidden_size)
        self.rssm = RSSM(hidden_size, action_size, hidden_size, stoch_size, stoch_classes)
        self.decoder = Decoder(state_size, obs_size, hidden_size)
        self.reward_model = RewardModel(state_size, hidden_size)
        self.continue_model = ContinueModel(state_size, hidden_size)

    def forward(self, observations, actions, rewards, continues):
        h, z = None, None
        all_h, all_z, all_prior, all_post = [], [], [], []
        for t in range(len(observations)):
            encoded = self.encoder(observations[t].symlog())
            h, z, prior, post = self.rssm.forward(h, z, actions[t], encoded_obs=encoded)
            all_h.append(h); all_z.append(z); all_prior.append(prior); all_post.append(post)
        H, Z = cat(all_h, axis=0), cat(all_z, axis=0)
        recon_loss = ((self.decoder(H, Z) - cat(observations, axis=0).symlog()) ** 2).mean()
        reward_logits = self.reward_model(H, Z)
        raw_rewards = _xp.asarray(cat(rewards, axis=0).data).reshape(-1)
        reward_loss = twohot_loss(reward_logits, raw_rewards).mean()
        cont_pred = self.continue_model(H, Z)
        cont_target = cat(continues, axis=0)
        continue_loss = (cont_target * cont_pred.log() * (-1.0) + (1.0 - cont_target) * (1.0 - cont_pred + Tensor(1e-8)).log() * (-1.0)).mean()
        prior_p = cat(all_prior, axis=0).softmax(axis=-1)
        post_p = cat(all_post, axis=0).softmax(axis=-1)
        kl_to_prior = (post_p.detach() * (post_p.detach().log() - prior_p.log())).sum(axis=-1).mean()
        kl_to_post = (post_p * (post_p.log() - prior_p.detach().log())).sum(axis=-1).mean()
        kl_loss = 0.5 * kl_to_prior.clamp(min_val=1.0) + 0.1 * kl_to_post.clamp(min_val=1.0)
        return 1.0 * recon_loss + 1.0 * reward_loss + 1.0 * continue_loss + kl_loss

    def parameters(self):
        p = self.encoder.parameters()
        p.extend(self.rssm.parameters())
        p.extend(self.decoder.parameters())
        p.extend(self.reward_model.parameters())
        p.extend(self.continue_model.parameters())
        return p

In [ ]:
%%writefile agent/actor.py
import numpy as np
from nn.mlp import MLP
from nn.tensor import Tensor, cat, _xp

class Actor:
    def __init__(self, state_size, action_size, hidden_size):
        self.action_size = action_size
        self.net = MLP([state_size, hidden_size, hidden_size, action_size * 2])
        self.net.layers[-1].weight.data *= 0.01
        if self.net.layers[-1].bias is not None:
            self.net.layers[-1].bias.data *= 0.0

    def __call__(self, h, z, deterministic=False):
        out = self.net(cat([h, z], axis=1))
        a = self.action_size
        # Split output into mean and log_std, BOTH connected to computation graph
        mean = Tensor(out.data[..., :a], requires_grad=out.requires_grad, _children=(out,), _op='split_mean')
        log_std_t = Tensor(out.data[..., a:], requires_grad=out.requires_grad, _children=(out,), _op='split_logstd')
        def _backward_mean():
            if out.requires_grad and mean.grad is not None:
                full_grad = _xp.zeros_like(out.data)
                full_grad[..., :a] = mean.grad
                out.grad = full_grad if out.grad is None else out.grad + full_grad
        def _backward_logstd():
            if out.requires_grad and log_std_t.grad is not None:
                full_grad = _xp.zeros_like(out.data)
                full_grad[..., a:] = log_std_t.grad
                out.grad = full_grad if out.grad is None else out.grad + full_grad
        mean._backward = _backward_mean
        log_std_t._backward = _backward_logstd
        log_std = log_std_t.clamp(min_val=-5.0, max_val=2.0)
        std = log_std.exp()
        if deterministic:
            return mean.tanh(), Tensor(0.0), Tensor(0.0)
        noise_np = np.random.randn(*std.data.shape).astype(np.float32)
        noise = _xp.asarray(noise_np) if hasattr(_xp, 'asarray') else noise_np
        pre_tanh = mean + std * Tensor(noise)
        action = pre_tanh.tanh()
        # Log prob of tanh-squashed Gaussian (differentiable through mean and log_std)
        gaussian_lp = (Tensor(noise) * Tensor(noise) * (-0.5) + log_std * (-1.0) +
                        Tensor(_xp.full(log_std.data.shape, -0.5 * np.log(2 * np.pi), dtype=_xp.float32)))
        squash_correction = (Tensor(_xp.ones_like(action.data)) + action * action * (-1.0) +
                             Tensor(_xp.full(action.data.shape, 1e-6, dtype=_xp.float32))).log() * (-1.0)
        log_prob = (gaussian_lp + squash_correction).sum(axis=-1, keepdims=True)
        entropy = (log_std + Tensor(_xp.full(log_std.data.shape, 0.5 * np.log(2 * np.pi * np.e), dtype=_xp.float32))).sum(axis=-1).mean()
        return action, log_prob, entropy

    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile agent/critic.py
import copy
from nn.mlp import MLP
from nn.linear import Linear, LayerNorm
from nn.tensor import Tensor, cat, _xp, NUM_BINS, BINS, symexp, twohot_decode

class Critic:
    def __init__(self, state_size, hidden_size):
        self.net = MLP([state_size, hidden_size, hidden_size, NUM_BINS])
        self.net.layers[-1].weight.data *= 0.01
        if self.net.layers[-1].bias is not None:
            self.net.layers[-1].bias.data *= 0.0
        self.target_params = [_xp.copy(p.data) for p in self.parameters()]

    def __call__(self, h, z):
        logits = self.net(cat([h, z], axis=1))
        return twohot_decode(logits)

    def logits(self, h, z):
        return self.net(cat([h, z], axis=1))

    def target_value(self, h, z):
        params = self.parameters()
        saved = [_xp.copy(p.data) for p in params]
        for p, tp in zip(params, self.target_params): p.data = tp
        logits = self.net(cat([h, z], axis=1))
        val = twohot_decode(logits)
        val = Tensor(val.data.copy())
        for p, s in zip(params, saved): p.data = s
        return val

    def update_target(self, tau=0.02):
        for i, p in enumerate(self.parameters()):
            self.target_params[i] = (1 - tau) * self.target_params[i] + tau * p.data

    def parameters(self): return self.net.parameters()

In [ ]:
%%writefile training/replay_buffer.py
import numpy as np
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.observations = deque(maxlen=capacity)
        self.actions = deque(maxlen=capacity)
        self.rewards = deque(maxlen=capacity)
        self.continues = deque(maxlen=capacity)
    def add(self, obs, action, reward, done):
        self.observations.append(obs); self.actions.append(action)
        self.rewards.append(reward); self.continues.append(0.0 if done else 1.0)
    def sample(self, batch_size, seq_len):
        max_start = len(self.observations) - seq_len
        if max_start < 1: return None
        idx = np.random.randint(0, max_start, size=batch_size)
        obs_s, act_s, rew_s, con_s = [], [], [], []
        for t in range(seq_len):
            obs_s.append(np.stack([self.observations[i+t] for i in idx]).astype(np.float32))
            act_s.append(np.stack([self.actions[i+t] for i in idx]).astype(np.float32))
            rew_s.append(np.stack([[self.rewards[i+t]] for i in idx]).astype(np.float32))
            con_s.append(np.stack([[self.continues[i+t]] for i in idx]).astype(np.float32))
        return obs_s, act_s, rew_s, con_s
    def __len__(self): return len(self.observations)

In [ ]:
# === SPEED TEST: CuPy vs NumPy ===
import sys, time
sys.path.insert(0, '/content')
import numpy as np
from nn.tensor import Tensor, GPU
from nn.mlp import MLP

print(f'GPU backend active: {GPU}')

net = MLP([376, 512, 512, 512])
x = Tensor(np.random.randn(32, 376).astype(np.float32), requires_grad=True)

# Warmup
out = net(x); out.sum().backward()

# Benchmark forward+backward
t0 = time.time()
for _ in range(50):
    for p in net.parameters(): p.zero_grad()
    x = Tensor(np.random.randn(32, 376).astype(np.float32), requires_grad=True)
    out = net(x)
    out.sum().backward()
elapsed = time.time() - t0
print(f'50 forward+backward passes: {elapsed:.2f}s ({elapsed/50*1000:.1f}ms each)')
print(f'Estimated epoch time (100 WM steps): {elapsed/50*100:.0f}s')

In [ ]:
import numpy as np
import time
import gymnasium as gym
from nn.tensor import Tensor, cat, to_cpu, _xp
from nn.optimizer import AdamW
from model.world_model import WorldModel
from agent.actor import Actor
from agent.critic import Critic
from training.replay_buffer import ReplayBuffer

# === CUSTOM HUMANOID WITH BETTER VISUALS ===
import gymnasium.envs.mujoco
import inspect, pathlib
src = pathlib.Path(inspect.getfile(gymnasium.envs.mujoco)).parent / 'assets' / 'humanoid.xml'
xml = open(src).read()
# Swap the brown/orange for a dark metallic blue body
xml = xml.replace('rgba=".7 .3 .1', 'rgba=".15 .25 .45')
xml = xml.replace('rgba=".8 .6 .4', 'rgba=".3 .4 .55')
xml = xml.replace('rgba=".5 .1 .1', 'rgba=".1 .15 .35')
# Brighter floor
xml = xml.replace('rgb1=".8 .9 .8"', 'rgb1=".85 .9 .95"')
xml = xml.replace('rgb2=".4 .6 .4"', 'rgb2=".55 .65 .75"')
with open(src, 'w') as f: f.write(xml)
print('Custom humanoid model saved (overwrote default XML)')

ENV_NAME = 'Humanoid-v4'
env = gym.make(ENV_NAME)
obs_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]
print(f'{ENV_NAME}: obs_size={obs_size}, action_size={action_size}')

hidden_size = 512
stoch_size = 32
stoch_classes = 32
state_size = hidden_size + stoch_size * stoch_classes

world_model = WorldModel(obs_size, action_size, hidden_size, stoch_size, stoch_classes)
actor = Actor(state_size, action_size, hidden_size=512)
critic = Critic(state_size, hidden_size=512)

wm_opt = AdamW(world_model.parameters(), lr=1e-4, grad_clip=1000.0)
actor_opt = AdamW(actor.parameters(), lr=3e-5, grad_clip=100.0)
critic_opt = AdamW(critic.parameters(), lr=8e-5, grad_clip=1000.0)
buffer = ReplayBuffer(capacity=100000)

# EMA percentile tracking (DreamerV3: running 5th/95th across batches)
ema_pct5 = None
ema_pct95 = None
ema_decay = 0.99

print(f'Params: WM={sum(p.data.size for p in world_model.parameters()):,} Actor={sum(p.data.size for p in actor.parameters()):,}')
print(f'State size: {state_size}')
print('Ready!')

In [ ]:
def collect_data(env, buffer, wm=None, actor=None, num_steps=1000, explore_noise=0.3):
    import nn.gru_cell as _gru_mod
    _gru_mod.INFERENCE_MODE = True
    obs, _ = env.reset()
    h, z = None, None
    prev_act = np.zeros(action_size, dtype=np.float32)
    for _ in range(num_steps):
        if actor and wm and np.random.rand() > 0.3:
            obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
            encoded = wm.encoder(obs_t.symlog())
            prev_a = Tensor(prev_act.reshape(1,-1))
            h, z, _, _ = wm.rssm.forward(h, z, prev_a, encoded_obs=encoded)
            h, z = h.detach(), z.detach()
            action, _, _ = actor(h, z)
            act_np = to_cpu(action.data)[0]
            act_np += np.random.randn(action_size).astype(np.float32) * explore_noise
            act_np = np.clip(act_np, -1.0, 1.0)
        else:
            act_np = env.action_space.sample()
        next_obs, reward, term, trunc, info = env.step(act_np)
        shaped_reward = reward + 5.0 * info.get('x_velocity', 0)
        buffer.add(obs, act_np, shaped_reward, term or trunc)
        prev_act = act_np.copy()
        obs = next_obs
        if term or trunc:
            obs, _ = env.reset()
            h, z = None, None
            prev_act = np.zeros(action_size, dtype=np.float32)
    _gru_mod.INFERENCE_MODE = False

def train_world_model(wm, opt, buffer, batch_size=16, seq_len=32):
    batch = buffer.sample(batch_size, seq_len)
    if batch is None: return None
    obs_s, act_s, rew_s, con_s = batch
    opt.zero_grad()
    loss = wm.forward([Tensor(o) for o in obs_s], [Tensor(a) for a in act_s],
                      [Tensor(r) for r in rew_s], [Tensor(c) for c in con_s])
    loss.backward()
    opt.step()
    return float(to_cpu(loss.data))

def train_actor_critic(wm, actor, critic, actor_opt, critic_opt, buffer, horizon=15, batch_size=32):
    GAMMA = 0.997
    LAMBDA = 0.95
    ENTROPY_SCALE = 3e-4
    batch = buffer.sample(batch_size, 1)
    if batch is None: return
    encoded = wm.encoder(Tensor(batch[0][0]).symlog())
    h, z, _, _ = wm.rssm.forward(None, None, Tensor(batch[1][0]), encoded_obs=encoded)
    h, z = h.detach(), z.detach()
    # === IMAGINATION: no h,z detach inside loop (dynamics backprop) ===
    imagined_h, imagined_z = [h], [z]
    rewards, continues, entropies = [], [], []
    for _ in range(horizon):
        action, _, entropy = actor(h, z)
        entropies.append(entropy)
        h, z, _, _ = wm.rssm.forward(h, z, action, encoded_obs=None)
        rewards.append(wm.reward_model.predict_diff(h, z))
        continues.append(wm.continue_model(h, z).detach())
        imagined_h.append(h.detach()); imagined_z.append(z.detach())
    # Target values from EMA critic (detached, no gradient)
    target_vals = [critic.target_value(imagined_h[t], imagined_z[t]) for t in range(horizon + 1)]
    # TD-lambda returns: rewards carry gradient, target values don't
    returns = [None] * horizon
    returns[-1] = rewards[-1] + continues[-1] * GAMMA * target_vals[horizon]
    for t in reversed(range(horizon - 1)):
        v_next = target_vals[t + 1]
        returns[t] = rewards[t] + continues[t] * GAMMA * (LAMBDA * returns[t + 1] + (1 - LAMBDA) * v_next)
    # === CRITIC LOSS: twohot cross-entropy (detached targets) ===
    from nn.tensor import twohot_loss as _twohot_loss
    critic_loss = Tensor(0.0)
    for t in range(horizon):
        logits_t = critic.logits(imagined_h[t], imagined_z[t])
        target_raw = returns[t].detach().data.reshape(-1)
        critic_loss = critic_loss + _twohot_loss(logits_t, target_raw).mean()
    critic_opt.zero_grad(); critic_loss.backward(); critic_opt.step()
    critic.update_target(tau=0.005)
    # === ACTOR LOSS: dynamics backprop with EMA percentile normalization ===
    global ema_pct5, ema_pct95
    all_returns = _xp.concatenate([r.data.flatten() for r in returns])
    batch_pct5 = float(_xp.percentile(all_returns, 5))
    batch_pct95 = float(_xp.percentile(all_returns, 95))
    if ema_pct5 is None:
        ema_pct5 = batch_pct5
        ema_pct95 = batch_pct95
    else:
        ema_pct5 = ema_decay * ema_pct5 + (1 - ema_decay) * batch_pct5
        ema_pct95 = ema_decay * ema_pct95 + (1 - ema_decay) * batch_pct95
    scale = max(1.0, ema_pct95 - ema_pct5)
    actor_loss = Tensor(0.0)
    for t in range(horizon):
        actor_loss = actor_loss + (returns[t] * (-1.0 / scale)).mean()
        actor_loss = actor_loss + (entropies[t] * (-ENTROPY_SCALE))
    actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()

def evaluate(env, wm, actor, num_episodes=2):
    import nn.gru_cell as _gru_mod
    _gru_mod.INFERENCE_MODE = True
    total = []
    for _ in range(num_episodes):
        obs, _ = env.reset()
        h, z = None, None
        prev_act = np.zeros(action_size, dtype=np.float32)
        ep_reward = 0
        for _ in range(500):
            obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
            encoded = wm.encoder(obs_t.symlog())
            prev_a = Tensor(prev_act.reshape(1,-1))
            h, z, _, _ = wm.rssm.forward(h, z, prev_a, encoded_obs=encoded)
            h, z = h.detach(), z.detach()
            action, _, _ = actor(h, z, deterministic=True)
            act_np = np.clip(to_cpu(action.data)[0], -1, 1)
            obs, reward, term, trunc, _ = env.step(act_np)
            prev_act = act_np.copy()
            ep_reward += reward
            if term or trunc: break
        total.append(ep_reward)
    _gru_mod.INFERENCE_MODE = False
    return sum(total)/len(total)

print('Functions defined!')

In [ ]:
# === QUICK TEST: 25 epochs to verify walking signal ===

print(f'Collecting initial random data...')
collect_data(env, buffer, num_steps=5000)
print(f'Buffer: {len(buffer)}')

baseline = evaluate(env, world_model, actor, num_episodes=3)
print(f'Before training (random): {baseline:.1f}')
print()

WM_WARMUP = 10
TOTAL_EPOCHS = 20
ACTOR_RATIO = 10  # Conservative: fewer actor updates to prevent collapse

print(f'--- QUICK TEST: {TOTAL_EPOCHS} epochs, WM warmup={WM_WARMUP} ---')
print()

best_reward = baseline
best_actor_params = [_xp.copy(p.data) for p in actor.parameters()]
best_critic_params = [_xp.copy(p.data) for p in critic.parameters()]
collapse_count = 0

start_time = time.time()
for epoch in range(TOTAL_EPOCHS):
    epoch_start = time.time()
    wm_losses = []
    train_actor = (epoch >= WM_WARMUP)

    for step in range(60):
        wm_loss = train_world_model(world_model, wm_opt, buffer)
        if wm_loss is not None: wm_losses.append(wm_loss)
        if train_actor and step % ACTOR_RATIO == 0:
            train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer)

    avg_wm = sum(wm_losses)/len(wm_losses) if wm_losses else 0
    phase = 'WM+Actor' if train_actor else 'WM only'
    avg_reward = evaluate(env, world_model, actor)

    # Checkpoint best policy
    if avg_reward > best_reward:
        best_reward = avg_reward
        best_actor_params = [_xp.copy(p.data) for p in actor.parameters()]
        best_critic_params = [_xp.copy(p.data) for p in critic.parameters()]
        collapse_count = 0

    # Rollback if collapsed (reward < 50% of best for 3 consecutive epochs)
    if train_actor and avg_reward < best_reward * 0.5:
        collapse_count += 1
        if collapse_count >= 3:
            print(f'  >> ROLLBACK: reward {avg_reward:.0f} < 50% of best {best_reward:.0f}, restoring best policy')
            for p, bp in zip(actor.parameters(), best_actor_params): p.data = _xp.copy(bp)
            for p, bp in zip(critic.parameters(), best_critic_params): p.data = _xp.copy(bp)
            collapse_count = 0
    else:
        collapse_count = max(0, collapse_count - 1)

    elapsed = time.time() - epoch_start
    total_t = time.time() - start_time
    print(f'epoch {epoch:2d} [{phase:9s}]: wm_loss={avg_wm:.4f}, reward={avg_reward:.1f}, '
          f'best={best_reward:.1f}, buffer={len(buffer)}, time={elapsed:.0f}s (total {total_t:.0f}s)')

    collect_data(env, buffer, world_model if train_actor else None,
                 actor if train_actor else None, num_steps=500, explore_noise=0.3)

print(f'\n=== QUICK TEST DONE ===')
print(f'Best reward: {best_reward:.1f}')
print(f'If best > 300, run FULL TRAINING cell below.')

In [ ]:
# === FULL TRAINING: Continues from quick test with checkpoint/rollback ===

FULL_EPOCHS = 100
ACTOR_RATIO = 15  # Conservative: 1 actor update per 15 WM steps

print(f'--- FULL TRAINING: {FULL_EPOCHS} more epochs (actor ratio 1:{ACTOR_RATIO}) ---')
print()

start_time = time.time()
collapse_count = 0

for epoch in range(FULL_EPOCHS):
    epoch_start = time.time()
    wm_losses = []

    for step in range(60):
        wm_loss = train_world_model(world_model, wm_opt, buffer)
        if wm_loss is not None: wm_losses.append(wm_loss)
        if step % ACTOR_RATIO == 0:
            train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer)

    avg_wm = sum(wm_losses)/len(wm_losses) if wm_losses else 0
    avg_reward = evaluate(env, world_model, actor)

    # Checkpoint best policy
    if avg_reward > best_reward:
        best_reward = avg_reward
        best_actor_params = [_xp.copy(p.data) for p in actor.parameters()]
        best_critic_params = [_xp.copy(p.data) for p in critic.parameters()]
        collapse_count = 0

    # Rollback on collapse (reward < 40% of best for 3 epochs)
    if avg_reward < best_reward * 0.4:
        collapse_count += 1
        if collapse_count >= 3:
            print(f'  >> ROLLBACK: reward {avg_reward:.0f} < 40% of best {best_reward:.0f}')
            for p, bp in zip(actor.parameters(), best_actor_params): p.data = _xp.copy(bp)
            for p, bp in zip(critic.parameters(), best_critic_params): p.data = _xp.copy(bp)
            collapse_count = 0
    else:
        collapse_count = max(0, collapse_count - 1)

    elapsed = time.time() - epoch_start
    total_t = time.time() - start_time
    print(f'epoch {epoch:2d}: wm_loss={avg_wm:.4f}, reward={avg_reward:.1f}, '
          f'best={best_reward:.1f}, buffer={len(buffer)}, time={elapsed:.0f}s (total {total_t:.0f}s)')

    collect_data(env, buffer, world_model, actor, num_steps=500, explore_noise=0.25)

# Restore best policy for final evaluation
for p, bp in zip(actor.parameters(), best_actor_params): p.data = _xp.copy(bp)
final = evaluate(env, world_model, actor, num_episodes=5)
print(f'\nFinal avg reward (best policy) = {final:.1f} (best seen: {best_reward:.1f})')
print(f'Total time: {time.time()-start_time:.1f}s')

In [ ]:
# === SAVE CHECKPOINT (run after training to preserve weights) ===
import pickle
checkpoint = {
    'wm': [to_cpu(p.data) for p in world_model.parameters()],
    'actor': [to_cpu(p.data) for p in actor.parameters()],
    'critic': [to_cpu(p.data) for p in critic.parameters()],
    'best_actor': [to_cpu(bp) for bp in best_actor_params],
    'best_critic': [to_cpu(bp) for bp in best_critic_params],
}
with open('checkpoint.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)
print(f'Checkpoint saved! (best_reward={best_reward:.1f})')
# Download from Colab: from google.colab import files; files.download('checkpoint.pkl')

In [ ]:
# === LOAD CHECKPOINT (skip training, go straight to render/diagnostics) ===
import pickle
with open('checkpoint.pkl', 'rb') as f:
    checkpoint = pickle.load(f)
for p, saved in zip(world_model.parameters(), checkpoint['wm']): p.data = _xp.array(saved)
for p, saved in zip(actor.parameters(), checkpoint['actor']): p.data = _xp.array(saved)
for p, saved in zip(critic.parameters(), checkpoint['critic']): p.data = _xp.array(saved)
best_actor_params = [_xp.array(bp) for bp in checkpoint['best_actor']]
best_critic_params = [_xp.array(bp) for bp in checkpoint['best_critic']]
print('Checkpoint loaded! Ready for render/diagnostics.')

In [ ]:
# === RENDER VIDEO (uses best saved policy) ===
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Restore best policy for rendering
import nn.gru_cell as _gru_mod
_gru_mod.INFERENCE_MODE = True
for p, bp in zip(actor.parameters(), best_actor_params): p.data = _xp.copy(bp)

render_env = gym.make(ENV_NAME, render_mode='rgb_array')
obs, _ = render_env.reset()
h, z = None, None
prev_act = np.zeros(action_size, dtype=np.float32)
frames = []
ep_reward = 0

for step in range(500):
    frames.append(render_env.render())
    obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
    encoded = world_model.encoder(obs_t.symlog())
    prev_a = Tensor(prev_act.reshape(1,-1))
    h, z, _, _ = world_model.rssm.forward(h, z, prev_a, encoded_obs=encoded)
    h, z = h.detach(), z.detach()
    action, _, _ = actor(h, z, deterministic=True)
    act_np = np.clip(to_cpu(action.data)[0], -1, 1)
    obs, reward, term, trunc, _ = render_env.step(act_np)
    prev_act = act_np.copy()
    ep_reward += reward
    if term or trunc: break

render_env.close()
print(f'Episode: {len(frames)} frames, reward={ep_reward:.1f}')

fig, ax = plt.subplots(figsize=(6,6))
ax.axis('off')
img = ax.imshow(frames[0])
def animate(i): img.set_data(frames[i]); return [img]
anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=33, blit=True)
plt.close()
_gru_mod.INFERENCE_MODE = False
HTML(anim.to_jshtml())

In [ ]:
# === OPTION C: IMAGINATION vs REALITY — World Model Diagnostics ===
# Side-by-side: what the world model predicts vs what actually happens
# Shows where the learned dynamics are accurate vs where they diverge

import mujoco
import nn.gru_cell as _gru_mod
_gru_mod.INFERENCE_MODE = True

IMAGINE_STEPS = 50  # How far ahead to imagine
START_STEP = 30     # Let the humanoid get moving first

# --- Phase 1: Run real trajectory, save states ---
diag_env = gym.make(ENV_NAME, render_mode='rgb_array')
obs, _ = diag_env.reset()
h, z = None, None
prev_act = np.zeros(action_size, dtype=np.float32)

# Run to START_STEP to get the humanoid moving
for _ in range(START_STEP):
    obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
    encoded = world_model.encoder(obs_t.symlog())
    prev_a = Tensor(prev_act.reshape(1,-1))
    h, z, _, _ = world_model.rssm.forward(h, z, prev_a, encoded_obs=encoded)
    h, z = h.detach(), z.detach()
    action, _, _ = actor(h, z, deterministic=True)
    act_np = np.clip(to_cpu(action.data)[0], -1, 1)
    obs, _, term, trunc, _ = diag_env.step(act_np)
    prev_act = act_np.copy()
    if term or trunc:
        obs, _ = diag_env.reset()
        h, z = None, None
        prev_act = np.zeros(action_size, dtype=np.float32)

# Save the branch point state
branch_h, branch_z = h.detach(), z.detach()
branch_qpos = diag_env.unwrapped.data.qpos.copy()
branch_qvel = diag_env.unwrapped.data.qvel.copy()
print(f'Branch point: step {START_STEP}, root pos = ({branch_qpos[0]:.2f}, {branch_qpos[1]:.2f}, {branch_qpos[2]:.2f})')

# --- Phase 2: Continue REAL trajectory for IMAGINE_STEPS ---
real_frames = []
real_obs_list = []
h_real, z_real = branch_h, branch_z
prev_act_real = prev_act.copy()
for t in range(IMAGINE_STEPS):
    real_frames.append(diag_env.render())
    real_obs_list.append(obs.copy())
    obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
    encoded = world_model.encoder(obs_t.symlog())
    prev_a = Tensor(prev_act_real.reshape(1,-1))
    h_real, z_real, _, _ = world_model.rssm.forward(h_real, z_real, prev_a, encoded_obs=encoded)
    h_real, z_real = h_real.detach(), z_real.detach()
    action, _, _ = actor(h_real, z_real, deterministic=True)
    act_np = np.clip(to_cpu(action.data)[0], -1, 1)
    obs, _, term, trunc, _ = diag_env.step(act_np)
    prev_act_real = act_np.copy()
    if term or trunc: break

# --- Phase 3: IMAGINED trajectory (no real env, pure world model) ---
imagined_frames = []
imagined_obs_list = []
h_im, z_im = branch_h, branch_z

# Create a separate env just for rendering imagined states
im_env = gym.make(ENV_NAME, render_mode='rgb_array')
im_env.reset()

# Track x,y position by integrating predicted velocity
im_x, im_y = branch_qpos[0], branch_qpos[1]
dt = im_env.unwrapped.model.opt.timestep * im_env.unwrapped.frame_skip

for t in range(len(real_frames)):
    # Actor chooses action from imagined state
    action, _, _ = actor(h_im, z_im, deterministic=True)
    # Step world model (no real environment!)
    h_im, z_im, _, _ = world_model.rssm.forward(h_im, z_im, action, encoded_obs=None)
    h_im, z_im = h_im.detach(), z_im.detach()
    # Decode imagined state back to observation
    decoded = world_model.decoder(h_im, z_im)
    pred_obs = to_cpu(decoded.data)[0]
    # Undo symlog: decoder predicts symlog'd obs
    pred_obs = np.sign(pred_obs) * (np.exp(np.abs(pred_obs)) - 1)
    imagined_obs_list.append(pred_obs.copy())
    # Reconstruct qpos from predicted observation
    pred_qpos = np.zeros(24)
    # Integrate position from predicted root velocity
    pred_vx = pred_obs[22]  # root x-velocity
    pred_vy = pred_obs[23]  # root y-velocity
    im_x += pred_vx * dt
    im_y += pred_vy * dt
    pred_qpos[0] = im_x
    pred_qpos[1] = im_y
    pred_qpos[2:] = pred_obs[0:22]  # z, quaternion, joints
    pred_qvel = pred_obs[22:45]
    # Set MuJoCo state and render
    im_env.unwrapped.data.qpos[:] = pred_qpos
    im_env.unwrapped.data.qvel[:] = pred_qvel
    mujoco.mj_forward(im_env.unwrapped.model, im_env.unwrapped.data)
    imagined_frames.append(im_env.render())

diag_env.close()
im_env.close()
_gru_mod.INFERENCE_MODE = False
print(f'Generated {len(real_frames)} real frames and {len(imagined_frames)} imagined frames')

# --- Phase 4: Side-by-side visualization ---
n_frames = min(len(real_frames), len(imagined_frames))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.set_title('REALITY', fontsize=14, fontweight='bold')
ax2.set_title('IMAGINATION (World Model)', fontsize=14, fontweight='bold')
ax1.axis('off'); ax2.axis('off')
img1 = ax1.imshow(real_frames[0])
img2 = ax2.imshow(imagined_frames[0])
plt.tight_layout()

def animate_compare(i):
    img1.set_data(real_frames[i])
    img2.set_data(imagined_frames[i])
    return [img1, img2]

anim = animation.FuncAnimation(fig, animate_compare, frames=n_frames, interval=50, blit=True)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# === DIVERGENCE ANALYSIS: per-joint prediction error over time ===

if real_obs_list and imagined_obs_list:
    n = min(len(real_obs_list), len(imagined_obs_list))
    real_arr = np.array(real_obs_list[:n])
    imag_arr = np.array(imagined_obs_list[:n])

    # Joint groups for analysis
    groups = {
        'Root (height/orient)': slice(0, 5),
        'Right Leg': slice(8, 12),
        'Left Leg': slice(12, 16),
        'Right Arm': slice(16, 19),
        'Left Arm': slice(19, 22),
        'Root Velocity': slice(22, 28),
    }

    fig, ax = plt.subplots(figsize=(10, 5))
    for name, idx in groups.items():
        error = np.abs(real_arr[:, idx] - imag_arr[:, idx]).mean(axis=1)
        ax.plot(error, label=name, linewidth=2)

    ax.set_xlabel('Imagination Step', fontsize=12)
    ax.set_ylabel('Mean Absolute Error', fontsize=12)
    ax.set_title('World Model Prediction Error by Body Part', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Summary stats
    print(f'\n--- Divergence Summary (over {n} steps) ---')
    for name, idx in groups.items():
        err = np.abs(real_arr[:, idx] - imag_arr[:, idx]).mean()
        err_final = np.abs(real_arr[-1, idx] - imag_arr[-1, idx]).mean()
        print(f'  {name:20s}: avg={err:.3f}, final_step={err_final:.3f}')
else:
    print('Run the imagination vs reality cell first!')